In [ ]:
import pandas as pd
import numpy as np
import yaml
import re
from typing import Dict, List, Tuple, Optional, Sequence

from pandas.api.types import is_string_dtype
from itertools import product

from pathlib import Path
from sklearn.metrics import fbeta_score
from training.predict import evaluate_models

In [ ]:
def classification_metrics_df(prepared_preds_df, id_col='unique_id', time_col='ds', target_col='y', beta: float = 1.0, overall_fbeta: bool = False):
    # basic validation
    if id_col not in prepared_preds_df.columns or time_col not in prepared_preds_df.columns or target_col not in prepared_preds_df.columns:
        raise ValueError(f"prepared_preds_df must contain columns {id_col}, {time_col}, {target_col}")

    # choose model columns automatically if not provided
    exclude = {id_col, time_col, target_col}
    models = [c for c in prepared_preds_df.columns if c not in exclude]
    if len(models) == 0:
        raise ValueError("No model columns found in prepared_preds_df")

    # directions to evaluate
    directions = prepared_preds_df[id_col].unique()

    results = []
    if overall_fbeta:
        # Compute overall fbeta for all directions combined
        y_true = (prepared_preds_df[target_col] > 0).astype(int)
        result_row = {"unique_id": "overall", "metric": f"fbeta_{beta}"}
        for model in models:
            y_pred = (prepared_preds_df[model] > 0).astype(int)
            fbeta = fbeta_score(y_true, y_pred, average='binary', zero_division=0, beta=beta)
            result_row[model] = fbeta
        results.append(result_row)

    for direction in directions:
        sub = prepared_preds_df[prepared_preds_df[id_col] == direction]
        # ground truth binary
        y_true = (sub[target_col] > 0).astype(int)
        result_row = {"unique_id": direction, "metric": f"fbeta_{beta}"}
        if not overall_fbeta:
            for model in models:
                y_pred = (sub[model] > 0).astype(int)
                fbeta = fbeta_score(y_true, y_pred, average='binary', zero_division=0, beta=beta)
                result_row[model] = fbeta
        results.append(result_row)
    return pd.DataFrame(results)

def evaluation_pipeline(pivoted_group: pd.DataFrame, beta: float = 1.0, overall_fbeta: bool = False) -> pd.DataFrame:
    evaluted_group_overall = evaluate_models(pivoted_group)
    evaluted_group_overall_to_keep = evaluted_group_overall[evaluted_group_overall["metric"] == "mae"].copy()
    evaluated_conditional_group = evaluate_models(pivoted_group[pivoted_group["y"] > 0])
    evaluated_conditional_group_to_keep = evaluated_conditional_group[evaluated_conditional_group["metric"] == "mae"].copy()
    evaluated_conditional_group_to_keep["metric"] = evaluated_conditional_group_to_keep["metric"].map({"mae": "mae_conditional"})
    evaluated_fbeta_group = classification_metrics_df(pivoted_group, beta=beta, overall_fbeta=overall_fbeta)
    evaluated_fbeta_group_to_keep = evaluated_fbeta_group[evaluated_fbeta_group["metric"] == f"fbeta_{beta}"].copy()
    evaluted_group = pd.concat([evaluted_group_overall_to_keep, evaluated_conditional_group_to_keep, evaluated_fbeta_group_to_keep], ignore_index=True)
    return evaluted_group

In [ ]:
def _parse_orderable_value(x):
    """
    Convert parameter values into something orderable.
    - numbers: keep as float/int
    - strings like '20_15_10': convert to tuple(int, int, int)
    - other strings: return as-is (lexicographic)
    """
    if pd.isna(x):
        return (np.inf,)  # push NaNs to the end
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if isinstance(x, str):
        parts = x.split("_")
        if all(re.fullmatch(r"-?\d+", p) for p in parts):
            return tuple(int(p) for p in parts)
        return x
    return x


def _lexicographic_simplicity_key(row: pd.Series,
                                 param_sort_rules: List[Tuple[str, bool]]):
    """
    Build a key tuple for lexicographic ordering based on (param, ascending_is_better).
    ascending_is_better=True means smaller values are preferred.
    """
    key_parts = []
    for param_name, ascending_is_better in param_sort_rules:
        v = _parse_orderable_value(row.get(param_name))
        # For descending preference, we invert the value if numeric/tuple-of-numeric.
        if ascending_is_better:
            key_parts.append(v)
        else:
            # invert for numeric
            if isinstance(v, (int, float)):
                key_parts.append(-v)
            # invert for tuple of ints
            elif isinstance(v, tuple) and all(isinstance(t, (int, float)) for t in v):
                key_parts.append(tuple(-t for t in v))
            else:
                # fallback: can't invert strings; keep as-is (rare for your params)
                key_parts.append(v)
    return tuple(key_parts)


def parameter_choice_with_guardrail_and_sorting(
    evaluation_dict: dict[str, pd.DataFrame], 
    model_name: str,
    primary_metric: str = "mae", 
    guardrail_metric: str = "fbeta_2.0", 
    primary_rel_tolerance: float = 0.02, 
    guardrail_rel_tolerance: float = 0.02, 
    tso_col: str = "tso", 
    direction_col: str = "unique_id", 
    pred_col: str = "y", 
    ds_col: str = "ds", 
    beta: float = 2.0, 
    higher_is_better_metrics: Sequence[str] = (), 
    weighted_average_metrics: Sequence[str] = (), 
    lexicographic_rules: Optional[list[tuple[str, bool]]] = None, 
): 
    """ For each group (e.g. each TSO), select the best parameter configuration based on a primary metric, with a guardrail metric to break ties. Then sort the selected configurations by simplicity based on lexicographic rules. 
    
    Parameters: 
    - evaluation_dict: dict of DataFrames containing evaluation results and volumes 
    - model_name: name of the column in DataFrames that contains the model name (e.g. "model") 
    - tso_col: name of the column that contains the group identifier (e.g. "TenneT DE") 
    - primary_metric: metric name to optimize (e.g. "mae") within `primary_rel_tolerance` 
    - guardrail_metric: metric name for tie-breaking (e.g. "fbeta_2.0") within `guardrail_rel_tolerance` 
    - weighted_average: whether to use a weighted average when computing metrics 
    - primary_rel_tolerance: relative tolerance for considering metrics as tied on primary metric (default 0.02 means within 2% of the best) 
    - guardrail_rel_tolerance: relative tolerance for considering metrics as tied on guardrail metric (default 0.02 means within 2% of the best) 
    - metric_col: name of the column in DataFrames that contains the metric names 
    - tso_col: name of the column that contains the group identifier (e.g. "tso") 
    - direction_col: name of the column that contains the direction/unique_id (e.g. "unique_id") 
    - pred_col: name of the column that contains the predicted values (e.g. "y") 
    - ds_col: name of the column that contains the timestamp (e.g. "ds") 
    - higher_is_better_metrics: list of metric names where higher values are better 
    - weighted_average_metrics: list of metric names for which to compute weighted average (requires 'volume' DataFrame in evaluation_dict) 
    - lexicographic_rules: optional lists of (param_name, ascending_is_better) for tie-breaking by simplicity 

    Returns: 
    - DataFrame with one row per group containing the selected parameter configuration and its metrics """ 
    if len(weighted_average_metrics) > 0 and "volumes" not in evaluation_dict: 
        raise ValueError("weighted_average=True requires 'volumes' DataFrame in evaluation_dict") 
    
    pred_df = evaluation_dict["predictions"] 
    volume_df = evaluation_dict.get("volumes", pd.DataFrame()) 
    results = [] 
    # Separate parameter columns 
    parameter_cols = [col for col in pred_df if col not in [direction_col, pred_col, tso_col, ds_col, model_name]] 
    metric_col = "metric" 
    for tso, group_df in pred_df.groupby(tso_col): 
        group_df = group_df.copy() 
        # Recompute metrics 
        eval_list = [] 
        for ablation_params, param_group in group_df.groupby(parameter_cols): 
            pivoted = evaluation_pipeline(param_group.drop(columns=[tso_col] + parameter_cols), beta=beta, overall_fbeta=True) 
            eval_list.append(pivoted.assign(**{tso_col: tso, **{col: val for col, val in zip(parameter_cols, ablation_params)}})) 
        eval_df = pd.concat(eval_list, ignore_index=True) 
        # If weighted average requested, merge with volume and compute weighted metrics 
        if len(weighted_average_metrics) > 0: 
            if volume_df.empty: 
                raise ValueError("Volume DataFrame is required for weighted average metrics") 
            volume_group = volume_df[volume_df[tso_col] == tso] 
            if volume_group.empty: 
                raise ValueError(f"No volume data for group {tso}") 
            eval_df = eval_df.merge(volume_group[[tso_col, direction_col, "total_volume"]], on=[tso_col, direction_col], how="left") 
            for metric in weighted_average_metrics: 
                if metric not in eval_df[metric_col].values: 
                    raise ValueError(f"Metric {metric} not found in evaluation DataFrame for group {tso}")
                metric_slice = eval_df[eval_df[metric_col] == metric].copy()
                weighted_metric = (
                    metric_slice.groupby(parameter_cols)[[model_name, "total_volume"]]
                    .apply(lambda x: np.average(x[model_name], weights=x["total_volume"]))
                    .reset_index(name=f"weighted_{metric}")
                )
                # Join the weighted metrics values for both directions back to eval_df 
                eval_df = eval_df.merge(weighted_metric, on=parameter_cols, how="left") 
                eval_df.loc[eval_df[metric_col] == metric, model_name] = eval_df.loc[eval_df[metric_col] == metric, f"weighted_{metric}"] 
                eval_df.drop(columns=[f"weighted_{metric}"], inplace=True, errors="ignore") 
            # Drop duplicates and the volume column 
            eval_df = eval_df.drop(columns=["total_volume"]) 
        # If somehow primary and guardrail are direction specific, average them and give a warning to the user 
        eval_df_tso_grouped = eval_df[eval_df[metric_col].isin([primary_metric, guardrail_metric])].groupby(parameter_cols + [metric_col])[model_name] 
        if eval_df_tso_grouped.nunique().max() > 1: 
            print(f"Warning: Metrics appear to be direction-specific for group {tso}. Averaging across directions for primary and guardrail metrics.") 
            eval_df = eval_df_tso_grouped.mean().reset_index() 
        else: 
            eval_df = eval_df_tso_grouped.first().reset_index() 
        # Pivot metrics to columes for easier filtering 
        eval_df = eval_df.pivot(index=parameter_cols, columns=metric_col, values=model_name).reset_index() 
        # Find the best primary metric value 
        if primary_metric in higher_is_better_metrics: 
            best_primary = eval_df[primary_metric].max() 
            primary_within_tol = eval_df[primary_metric] >= best_primary * (1.0 - primary_rel_tolerance) 
        else: 
            best_primary = eval_df[primary_metric].min() 
            primary_within_tol = eval_df[primary_metric] <= best_primary * (1.0 + primary_rel_tolerance) 
        # Now among those within tolerance on primary, find the best guardrail metric 
        primary_survivors = eval_df[primary_within_tol].copy() 
        if guardrail_metric in higher_is_better_metrics: 
            best_guardrail = primary_survivors[guardrail_metric].max() 
            guardrail_within_tol = primary_survivors[guardrail_metric] >= best_guardrail * (1.0 - guardrail_rel_tolerance) 
        else: 
            best_guardrail = primary_survivors[guardrail_metric].min() 
            guardrail_within_tol = primary_survivors[guardrail_metric] <= best_guardrail * (1.0 + guardrail_rel_tolerance) 
        final_candidates = primary_survivors[guardrail_within_tol].copy() 
        # If not candidate exists,fall back to primary-only survivors 
        if final_candidates.empty: 
            final_candidates = eval_df[primary_within_tol].copy() 
        # If still empty (should not happen), fall back to absolute best primary 
        if final_candidates.empty: 
            if primary_metric in higher_is_better_metrics: 
                final_candidates = eval_df.loc[[eval_df[primary_metric].idxmax()]].copy() 
            else: 
                final_candidates = eval_df.loc[[eval_df[primary_metric].idxmin()]].copy() 
        # If lexicographic rules are provided for this model, sort by them 
        if lexicographic_rules: 
            final_candidates["_simplicity_key"] = final_candidates.apply(lambda row: _lexicographic_simplicity_key(row, lexicographic_rules), axis=1) 
            chosen_row = final_candidates.sort_values(by="_simplicity_key", ascending=True).iloc[0] 
            final_candidates.drop(columns=["_simplicity_key"], inplace=True, errors="ignore") 
        else: 
            # If no rules given: choose smallest primary metric, then smallest guardrail, then config columns 
            sort_cols = [primary_metric, guardrail_metric] + parameter_cols 
            ascending = [ primary_metric not in higher_is_better_metrics, guardrail_metric not in higher_is_better_metrics, ] + [True] * len(parameter_cols) 
            chosen_row = final_candidates.sort_values(by=sort_cols, ascending=ascending).iloc[0] 
        out = {"model": model_name, "tso": tso, **{col: chosen_row[col] for col in parameter_cols}} 
        out[f"mean_{primary_metric}"] = chosen_row[primary_metric] 
        out[f"mean_{guardrail_metric}"] = chosen_row[guardrail_metric] 
        out["best_mean_" + primary_metric] = best_primary 
        out["best_mean_" + guardrail_metric] = best_guardrail 
        out["primary_rel_tolerance"] = primary_rel_tolerance 
        out["guardrail_rel_tolerance"] = guardrail_rel_tolerance 
        results.append(out) 

    return pd.DataFrame(results)


# Window sparsity

In [ ]:
window_data = pd.concat([
    pd.read_csv("input_size_ablation_paper/50hertz/validation_predictions_basic_day_ahead_price_wind_pv_production_consumption_sce_50Hertz_k2_checkpoint_best.csv", parse_dates=["ds"]),
    pd.read_csv("input_size_ablation_paper/amprion/validation_predictions_basic_day_ahead_price_wind_pv_production_consumption_sce_Amprion_k2_checkpoint_best.csv", parse_dates=["ds"]),
    pd.read_csv("input_size_ablation_paper/tennet_de/validation_predictions_basic_day_ahead_price_wind_pv_production_consumption_sce_TenneT_DE_k2_checkpoint_best.csv", parse_dates=["ds"]),
    pd.read_csv("input_size_ablation_paper/transnetbw/validation_predictions_basic_day_ahead_price_wind_pv_production_consumption_sce_TransnetBW_k2_checkpoint_best.csv", parse_dates=["ds"]),
], axis=0)
window_data.drop_duplicates(subset=["tso", "ds", "unique_id", "window_index"]).groupby(["tso", "unique_id"])["y"].apply(
    lambda g: (g == 0.).sum() / len(g)
)

# `input_size` Ablation

In [ ]:
def process_input_size_predictions(root_dir: Path, dataset_name: str, beta: float = 1.0, overlap_agg_func: str = "mean"):
    all_dataset_evaluations = []
    for input_size_file in root_dir.glob(f"**/validation_predictions_{dataset_name}*.csv"):
        df = pd.read_csv(input_size_file, parse_dates=["ds"]).drop(columns=[
            "model_alias", "val_start", "val_end", "horizon"
        ])
        tso_evaluations = []
        tso_name = df["tso"].iloc[0]
        # Prepare to the correct model specific format
        for index, group in df.groupby(["window_index", "input_size"]):
            # Because eval windows overlap, we can have multiple rows per (ds, unique_id) with different y_hat from different windows. 
            # We aggregate them before evaluation.
            pivoted_group = group.pivot_table(
                index=["ds", "y", "unique_id"], columns="model_name", values="y_hat",
                aggfunc=overlap_agg_func
            ).reset_index()
            pivoted_group.columns.name = None
            evaluated_group = evaluation_pipeline(pivoted_group, beta=beta)
            tso_evaluations.append(evaluated_group.assign(window=index[0], input_size=index[1], tso=tso_name))

        all_dataset_evaluations.append(pd.concat(tso_evaluations, ignore_index=True))

    return pd.concat(all_dataset_evaluations, ignore_index=True)


def process_input_size_predictions_no_window(root_dir: Path, dataset_name: str, beta: float = 1.0):
    all_dataset_evaluations = []
    all_dataset_volumes = []
    for input_size_file in root_dir.glob(f"**/validation_predictions_{dataset_name}*.csv"):
        df = pd.read_csv(input_size_file, parse_dates=["ds"]).drop(columns=[
            "model_alias", "val_start", "val_end", "horizon"
        ])
        tso_evaluations = []
        tso_name = df["tso"].iloc[0]
        # Prepare to the correct model specific format
        for index, group in df.groupby(["input_size"]):
            # Because eval windows overlap, we can have multiple rows per (ds, unique_id) with different y_hat from different windows. 
            # We aggregate them before evaluation.
            pivoted_group = group.pivot_table(
                index=["ds", "y", "unique_id"], columns="model_name", values="y_hat",
                aggfunc="mean"
            ).reset_index()
            pivoted_group.columns.name = None
            evaluated_group = evaluation_pipeline(pivoted_group, beta=beta)
            volumes_group = group.groupby("unique_id")["y"].sum().reset_index().assign(tso=tso_name).rename(
                columns={"y": "total_volume"}
            )
            all_dataset_volumes.append(volumes_group)
            tso_evaluations.append(evaluated_group.assign(input_size=index[0], tso=tso_name))

        all_dataset_evaluations.append(pd.concat(tso_evaluations, ignore_index=True))

    concat_evalatuions = pd.concat(all_dataset_evaluations, ignore_index=True)
    concat_volumes = pd.concat(all_dataset_volumes, ignore_index=True).drop_duplicates(subset=["tso", "unique_id"])
    return concat_evalatuions, concat_volumes

In [ ]:
input_size_evaluations_no_window, input_size_volumes = process_input_size_predictions_no_window(
    root_dir=Path("/home/jovyan/redispatch-ml-forecasting/input_size_ablation_paper/"),
    dataset_name="basic_day_ahead_price_wind_pv_production_consumption_sce",
    beta=2.0
)

In [ ]:
input_size_evaluations_formatted = input_size_evaluations_no_window.set_index([
    "tso", "unique_id", "metric", "input_size"
]).drop(columns=["merge_key"]).unstack(level=3).loc[(slice(None), slice(None), ["mae", "mae_conditional", "fbeta_2.0"], slice(None)), :]

In [ ]:
input_size_evaluations_formatted.loc[("50Hertz", "down", slice(None), slice(None)), :]#.to_csv("input_size_ablation/50Hertz_evaluation.csv")

In [ ]:
input_size_evaluations_mae_ranked = input_size_evaluations_no_window.query("metric == 'mae'").drop(columns=["metric", "merge_key"]).set_index(["tso", "unique_id", "input_size"])
input_size_evaluations_fbeta_ranked = input_size_evaluations_no_window.query("metric == 'fbeta_2.0'").drop(columns=["metric", "merge_key"]).set_index(["tso", "unique_id", "input_size"])

In [ ]:
input_size_mae_rankings = input_size_evaluations_mae_ranked.stack().groupby(level=[0, 1, 3]).rank().groupby(level=["tso", "input_size"]).mean()

In [ ]:
input_size_mae_rankings.unstack(level=0)

In [ ]:
input_size_mae_rankings.unstack(level=0).to_latex("input_size_ablation_paper/input_size_mae_rankings.tex", float_format="%.2f", multicolumn=True, multirow=True, caption="Average MAE rankings across TSOs and directions for different input sizes.", label="tab:input_size_mae_rankings")

In [ ]:
input_size_fbeta_rankings = input_size_evaluations_fbeta_ranked.stack().groupby(level=[0, 1, 3]).rank(ascending=False).groupby(level=["tso", "input_size"]).mean()

In [ ]:
(input_size_mae_rankings * 0.75).add(input_size_fbeta_rankings * 0.25, fill_value=0)

In [ ]:
(input_size_evaluations_mae_ranked.stack().groupby(level=[0, 1, 3]).rank() == 1).groupby(level=["tso", "input_size"]).mean().unstack(level=0)

# Neural model ablation

In [ ]:
def extract_model_parameteres_from_label(model_alias: str, model_name: str, input_size: int, sweep_data: dict) -> dict:
    input_size_text = rf"{model_name}_i{input_size}_"
    model_parameters_string = model_alias.replace(input_size_text, "")
    found_parameters = {}
    for parameter_name, parameter_values in sweep_data[model_name].items():
        # Find the parameter value in the model alias string
        for value in parameter_values:
            if isinstance(value, list):
                value_text = f"{parameter_name}_{'_'.join(map(str, value))}"
            elif isinstance(value, float):
                converted_float_to_text = str(value).replace(".", "p").replace("-", "m")
                value_text = f"{parameter_name}_{converted_float_to_text}"
            elif isinstance(value, int):
                value_text = f"{parameter_name}_{value}"
            else:
                raise ValueError(f"Unsupported parameter value type: {type(value)} for parameter {parameter_name} with value {value}")

            if value_text in model_parameters_string:
                if isinstance(value, list):
                    found_parameters[parameter_name] = '_'.join(map(str, value))
                else:
                    found_parameters[parameter_name] = value
    return found_parameters


def process_input_size_predictions_neural_ablation(root_dir: Path, ablation_sweep_path: Path, dataset_name: str, overlap_agg: str = "mean"):
    all_model_evaluations = {}
    for input_size_file in root_dir.glob(f"validation_predictions_{dataset_name}*.csv"):
        df = pd.read_csv(input_size_file, parse_dates=["ds"]).drop(columns=[
            "val_start", "val_end", "horizon"
        ])
        # tso_evaluations = []
        tso_name = df["tso"].iloc[0]
        # Extract ablation parameters using the sweep file
        with open(ablation_sweep_path, "r") as f:
            sweep_data = yaml.safe_load(f)

        # Prepare to the correct model specific format
        for (w, input_size, model_alias), group in df.groupby(["window_index", "input_size", "model_alias"]):
            # Because eval windows overlap, we can have multiple rows per (ds, unique_id) with different y_hat from different windows. 
            # We aggregate them before evaluation.
            pivoted_group = group.pivot_table(
                index=["ds", "y", "unique_id"], columns="model_name", values="y_hat",
                aggfunc=overlap_agg
            ).reset_index()
            pivoted_group.columns.name = None
            evaluted_group = evaluate_models(pivoted_group)
            model_name = group["model_name"].iloc[0]
            if model_name in sweep_data:
                # Extract ablation parameters for this model
                ablation_parameters = extract_model_parameteres_from_label(model_alias, model_name, input_size, sweep_data)
                # print(model_alias)
                # print(ablation_parameters)
                prepared_evaluation = evaluted_group.assign(window=w, input_size=input_size, tso=tso_name, **ablation_parameters)

        # all_model_evaluations.append(pd.concat(tso_evaluations, ignore_index=True))
            all_model_evaluations.setdefault(model_name, []).append(prepared_evaluation)

    concat_model_evaluations = {model: pd.concat(evals, ignore_index=True) for model, evals in all_model_evaluations.items()}

    return concat_model_evaluations

def process_input_size_predictions_neural_ablation_no_window(root_dir: Path, ablation_sweep_path: Path, dataset_name: str, beta: float = 1.0, overlap_agg: str = "mean"):
    all_model_evaluations = {}
    all_model_volumes = {}
    all_model_predictions = {}
    for input_size_file in root_dir.glob(f"validation_predictions_{dataset_name}*.csv"):
        df = pd.read_csv(input_size_file, parse_dates=["ds"]).drop(columns=[
            "val_start", "val_end", "horizon"
        ])
        # tso_evaluations = []
        tso_name = df["tso"].iloc[0]
        # Extract ablation parameters using the sweep file
        with open(ablation_sweep_path, "r") as f:
            sweep_data = yaml.safe_load(f)

        # Prepare to the correct model specific format
        for (input_size, model_alias), group in df.groupby(["input_size", "model_alias"]):
            # Because eval windows overlap, we can have multiple rows per (ds, unique_id) with different y_hat from different windows. 
            # We aggregate them before evaluation.
            pivoted_group = group.pivot_table(
                index=["ds", "y", "unique_id"], columns="model_name", values="y_hat",
                aggfunc=overlap_agg
            ).reset_index()
            pivoted_group.columns.name = None
            evaluted_group = evaluation_pipeline(pivoted_group.copy(deep=True), beta=beta)
            volumes_group = group.groupby("unique_id")["y"].sum().reset_index().assign(tso=tso_name).rename(
                columns={"y": "total_volume"}
            )
            model_name = group["model_name"].iloc[0]
            if model_name in sweep_data:
                # Extract ablation parameters for this model
                ablation_parameters = extract_model_parameteres_from_label(model_alias, model_name, input_size, sweep_data)
                if not ablation_parameters:
                    continue
                prepared_evaluation = evaluted_group.assign(input_size=input_size, tso=tso_name, **ablation_parameters)
                prepared_predictions = pivoted_group.assign(input_size=input_size, tso=tso_name, **ablation_parameters)

            all_model_evaluations.setdefault(model_name, []).append(prepared_evaluation)
            all_model_volumes.setdefault(model_name, []).append(volumes_group)
            all_model_predictions.setdefault(model_name, []).append(prepared_predictions)

    return {
        model: {
            "evaluations": pd.concat(evals, ignore_index=True),
            "volumes": pd.concat(vols, ignore_index=True),
            "predictions": pd.concat(preds, ignore_index=True)
        } for model, (evals, vols, preds) in zip(all_model_evaluations.keys(), zip(all_model_evaluations.values(), all_model_volumes.values(), all_model_predictions.values()))
    }

def assemble_neural_ablation_results_no_window(ablation_sweep_paths_per_tso: dict[Path, list[str]], ablation_sweep_path: Path, dataset_name: str, beta: float = 1.0):
    final_ablations_dict = {}
    prediction_ablation_dict = {}
    volume_ablations_dict = {}
    for ablation_dir, relevant_tsos in ablation_sweep_paths_per_tso.items():
        evaluations_dict = process_input_size_predictions_neural_ablation_no_window(
            root_dir=ablation_dir,
            ablation_sweep_path=ablation_sweep_path,
            dataset_name=dataset_name,
            beta=beta
        )
        for model_name, eval_dict in evaluations_dict.items():
            eval_df = eval_dict["evaluations"]
            volumes_df = eval_dict["volumes"]
            predictions_df = eval_dict["predictions"]
            eval_df_tso_filtered = eval_df[eval_df["tso"].isin(relevant_tsos)].copy()
            volume_df_tso_filtered = volumes_df[volumes_df["tso"].isin(relevant_tsos)].copy()
            prediction_df_tso_filtered = predictions_df[predictions_df["tso"].isin(relevant_tsos)].copy()
            final_ablations_dict.setdefault(model_name, []).append(eval_df_tso_filtered)
            volume_ablations_dict.setdefault(model_name, []).append(volume_df_tso_filtered)
            prediction_ablation_dict.setdefault(model_name, []).append(prediction_df_tso_filtered)

    final_ablations_concat = {
        model: {
            "evaluations": pd.concat(dfs, ignore_index=True),
            "volumes": pd.concat(vols, ignore_index=True).drop_duplicates(subset=["tso", "unique_id"]),
            "predictions": pd.concat(preds, ignore_index=True)
        } for model, (dfs, vols, preds) in zip(final_ablations_dict.keys(), zip(final_ablations_dict.values(), volume_ablations_dict.values(), prediction_ablation_dict.values()))
    }

    return final_ablations_concat

In [ ]:
best_ablation_sweep_paths_per_tso = {
    Path("neural_parameter_ablation_paper/"): [
        "TenneT DE", "TransnetBW", "Amprion", "50Hertz",
    ],
}

neural_ablation_evaluations_combined_no_window = assemble_neural_ablation_results_no_window(
    ablation_sweep_paths_per_tso=best_ablation_sweep_paths_per_tso,
    ablation_sweep_path=Path("training/ablation_neural_config.yaml"),
    dataset_name="basic_day_ahead_price_wind_pv_production_consumption_sce",
    beta=2.0,
)

In [ ]:
neural_ablation_evaluations_combined_no_window["nbeatsx"]["evaluations"].drop(columns=["n_blocks"])

In [ ]:
nbeatsx_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=neural_ablation_evaluations_combined_no_window["nbeatsx"],
    model_name="nbeatsx",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("n_blocks", True)]
)

In [ ]:
nbeatsx_ablation_choice

In [ ]:
nhits_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=neural_ablation_evaluations_combined_no_window["nhits"],
    model_name="nhits",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("n_blocks", True)]
)

In [ ]:
nhits_ablation_choice

In [ ]:
tft_mae_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=neural_ablation_evaluations_combined_no_window["tft"],
    model_name="tft",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("hidden_size", True), ("dropout", False)]
)

In [ ]:
tft_mae_ablation_choice

In [ ]:
lstm_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=neural_ablation_evaluations_combined_no_window["lstm"],
    model_name="lstm",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("encoder_hidden_size", True), ("encoder_dropout", False)]
)

In [ ]:
lstm_ablation_choice

# Benchmark ablation

In [ ]:
def extract_benchmark_model_parameters_from_label(model_alias: str, model_name: str) -> dict:
    input_size_text = rf"{model_name}"
    model_parameters_string = model_alias.replace(input_size_text, "")
    found_parameters = {}
    if "arima" in model_name.lower():
        order_params = {}
        for order_name in ["p", "d", "q"]:
            match = re.search(rf"{order_name}(\d+)", model_parameters_string)
            if match:
                order_params[order_name] = int(match.group(1))
            else:
                order_params[order_name] = 0
        found_parameters.update(order_params)
        seasonality_parms = {}
        for seasonality_name in ["P", "D", "Q", "m"]:
            match = re.search(rf"{seasonality_name}(\d+)", model_parameters_string)
            if match:
                seasonality_parms[seasonality_name] = int(match.group(1))
            else:
                seasonality_parms[seasonality_name] = 0
        found_parameters.update(seasonality_parms)
    elif model_name.lower() == "lightgbm":
        match = re.search(r"lightgbm_mldl_(\d+)", model_alias)
        if match:
            found_parameters["min_data_in_leaf"] = int(match.group(1))
        else:
            found_parameters["min_data_in_leaf"] = None
    elif model_name.lower() == "ridge":
        match = re.search(r"ridge_alpha_([\dmp]+)", model_alias)
        if match:
            alpha_d_str = match.group(1)
            alpha_value = float(alpha_d_str.replace("p", ".").replace("m", "-"))
            found_parameters["alpha"] = alpha_value
        else:
            found_parameters["alpha"] = None
    elif model_name.lower() == "lasso":
        match = re.search(r"lasso_alpha_([\dmp]+)", model_alias)
        if match:
            alpha_d_str = match.group(1)
            alpha_value = float(alpha_d_str.replace("p", ".").replace("m", "-"))
            found_parameters["alpha"] = alpha_value
        else:
            found_parameters["alpha"] = None
    elif model_name.lower() == "elasticnet":
        match = re.search(r"elasticnet_alpha_([\dmp]+)_l1r([\dmp]+)", model_alias)
        if match:
            alpha_d_str = match.group(1)
            alpha_value = float(alpha_d_str.replace("p", ".").replace("m", "-"))
            found_parameters["alpha"] = alpha_value
            l1_ratio_str = match.group(2)
            l1_ratio_value = float(l1_ratio_str.replace("p", ".").replace("m", "-"))
            found_parameters["l1_ratio"] = l1_ratio_value
        else:
            found_parameters["alpha"] = None
            found_parameters["l1_ratio"] = None
    elif model_name.lower() == "croston":
        match = re.search(r"croston_tsb_ad([\dmp]+)_ap([\dmp]+)", model_alias)
        if match:
            alpha_d_str = match.group(1)
            alpha_d = float(alpha_d_str.replace("p", ".").replace("m", "-"))
            found_parameters["alpha_d"] = alpha_d
            alpha_p_str = match.group(2)
            alpha_p = float(alpha_p_str.replace("p", ".").replace("m", "-"))
            found_parameters["alpha_p"] = alpha_p
        else:
            found_parameters["alpha_d"] = None
            found_parameters["alpha_p"] = None
    else:
        raise ValueError(f"Unknown model name '{model_name}' for parameter extraction.")

    return found_parameters


def read_benchmark_ablation_results_no_window(root_dir: Path, dataset_name: str, overlap_agg: str = "mean", beta: float = 1.0, exclude_arima: bool = True):
    evalations_per_model = {}
    tso_volumes = {}
    predictions_per_model = {}
    for file in root_dir.glob(f"**/benchmark_ablation_{dataset_name}*.csv"):
        df = pd.read_csv(file, parse_dates=["ds"]).drop(columns=[
            "val_start", "val_end", "horizon"
        ])
        tso_name = df["tso"].iloc[0]
        for (input_size, model_alias), group in df.groupby(["input_size", "model_alias"]):
            model_name = group["model_name"].iloc[0]
            if not exclude_arima or "arima" not in model_name.lower():
                pivoted_group = group.pivot_table(
                    index=["ds", "y", "unique_id"], columns="model_alias", values="y_hat", aggfunc=overlap_agg
                ).reset_index().rename(
                    columns={model_alias: model_name}
                )
                pivoted_group.columns.name = None
                model_parameters = extract_benchmark_model_parameters_from_label(
                    model_alias, 
                    model_name=model_name,
                )
                volumes = group.groupby("unique_id")["y"].sum().reset_index().assign(tso=tso_name).rename(
                    columns={"y": "total_volume"}
                )
                model_predictions = pivoted_group.assign(input_size=input_size, tso=tso_name, **model_parameters)
                evaluated_group = evaluation_pipeline(pivoted_group, beta=beta).assign(input_size=input_size, tso=tso_name, **model_parameters)
                evalations_per_model.setdefault(model_name, []).append(evaluated_group)
                predictions_per_model.setdefault(model_name, []).append(model_predictions)
                tso_volumes.setdefault(model_name, []).append(volumes) 

    # return {
    #     model: pd.concat(all_evaluations, ignore_index=True)
    #     for model, all_evaluations in evalations_per_model.items()
    # }, pd.concat(tso_volumes.values(), ignore_index=True)
    return {
        model: {
            "evaluations": pd.concat(all_evaluations, ignore_index=True),
            "predictions": pd.concat(all_predictions, ignore_index=True),
            "volumes": pd.concat(all_volumes, ignore_index=True).drop_duplicates(subset=["tso", "unique_id"])
        } 
        for model, (all_evaluations, all_predictions, all_volumes) in zip(evalations_per_model.keys(), zip(evalations_per_model.values(), predictions_per_model.values(), tso_volumes.values()))
    }

In [ ]:
benchmark_ablation_evaluations = read_benchmark_ablation_results_no_window(
    root_dir=Path("benchmark_ablation_paper/"),
    dataset_name="basic_day_ahead_price_wind_pv_production_consumption_sce",
    beta=2.0,
)

benchmark_ablation_linear_evaluations = read_benchmark_ablation_results_no_window(
    root_dir=Path("benchmark_ablation_paper_linear/"),
    dataset_name="basic_day_ahead_price_wind_pv_production_consumption_sce",
    beta=2.0,
)

In [ ]:
ridge_non_scaled_benchmark_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=benchmark_ablation_evaluations["ridge"],
    model_name="ridge",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("alpha", False)]
)

In [ ]:
ridge_non_scaled_benchmark_ablation_choice

In [ ]:
ridge_scaled_benchmark_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=benchmark_ablation_linear_evaluations["ridge"],
    model_name="ridge",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("alpha", False)]
)

In [ ]:
ridge_scaled_benchmark_ablation_choice

In [ ]:
lasso_scaled_benchmark_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=benchmark_ablation_linear_evaluations["lasso"],
    model_name="lasso",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("alpha", False)]
)

In [ ]:
lasso_scaled_benchmark_ablation_choice

In [ ]:
elasticnet_scaled_benchmark_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=benchmark_ablation_linear_evaluations["elasticnet"],
    model_name="elasticnet",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("alpha", False), ("l1_ratio", True)]
)

In [ ]:
elasticnet_scaled_benchmark_ablation_choice

In [ ]:
lightgbm_benchmark_ablation_choice = parameter_choice_with_guardrail_and_sorting(
    evaluation_dict=benchmark_ablation_evaluations["lightgbm"],
    model_name="lightgbm",
    primary_metric="mae", guardrail_metric="fbeta_2.0",
    primary_rel_tolerance=0.02, guardrail_rel_tolerance=0.02, 
    higher_is_better_metrics=("fbeta_2.0",), weighted_average_metrics=("mae",),
    lexicographic_rules=[("min_data_in_leaf", False)]
)

In [ ]:
lightgbm_benchmark_ablation_choice